# Generate MongoDB queries with LLMs

This notebook walks through the JSON-Storm generation pipeline:

**example documents → BONDI schemas → schema text → prompt → model response → parsed queries**

We use small synthetic documents from three related collections. No MongoDB connection is required, and no queries are executed. The model-call cell uses your local Ollama `qwen3.6:latest` by default. Run cells in order; pause to inspect each intermediate representation.

## 1. Set up the project environment

From the repository root, run `uv sync --locked`, then select the project's `.venv` Python kernel in your notebook editor. Start Ollama and ensure `qwen3.6:latest` is available before running generation.

The setup below locates the repository whether the kernel starts in the root or `notebooks/`. It loads `.env` only if present, without overriding existing environment variables. Ollama does not require provider credentials.

In [ ]:
from pathlib import Path

import pandas as pd
from bondi.schema import Schema
from bondi.workload import Workload, WorkloadEntry
from IPython.display import display

from json_storm.generation import generate_queries
from json_storm.models import LiteLLMModel, load_credentials
from json_storm.prompts import PROMPTS
from json_storm.schema import render_schema_catalog

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/json_storm/generation.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Start this notebook from the JSON-Storm repository.")
if (PROJECT_ROOT / ".env").is_file():
    load_credentials(PROJECT_ROOT / ".env")
print(f"Project: {PROJECT_ROOT}")

Project: /Users/thomas/code/bondi-json-storm


## 2. Inspect example documents with BONDI

Each collection gets its own `Schema`. Notice the nested address, array-valued categories, missing city, and explicit null rating. These observations describe only the supplied documents—not every possible document in a collection.

`values="distinct"` retains primitive examples so we can optionally show them later. It does **not** automatically include them in the prompt. On a real dataset, obtain a bounded sample before constructing these schemas.

In [ ]:
documents = {
    "businesses": [
        {
            "business_id": "b1",
            "name": "Harbour Cafe",
            "stars": 4.5,
            "address": {"city": "Sydney"},
            "categories": ["Cafe", "Breakfast"],
        },
        {"business_id": "b2", "name": "Park Bakery", "stars": None, "address": {}, "categories": ["Bakery"]},
    ],
    "reviews": [
        {"business_id": "b1", "user_id": "u1", "rating": 5, "text": "Great coffee"},
        {"business_id": "b2", "user_id": "u2", "rating": 3, "text": "Good bread"},
    ],
    "users": [
        {"user_id": "u1", "name": "Alex", "review_count": 12},
        {"user_id": "u2", "name": "Sam", "review_count": 3},
    ],
}
schemas = {name: Schema.from_documents(rows, values="distinct") for name, rows in documents.items()}
for name, schema in schemas.items():
    print(f"{name}: {schema.document_count} inspected documents")

businesses: 2 inspected documents
reviews: 2 inspected documents
users: 2 inspected documents


## 3. Render the schema catalogue

Explicit paths make nesting visible: `address.city` is nested, `categories[]` describes array elements, and a quoted path like `["address.city"]` would mean a literal field name containing a dot. The array notation is descriptive, not MongoDB query syntax.

Leave examples disabled for the baseline. Try `EXAMPLES_PER_TYPE = 2` to see how observed values change the prompt; this can affect whether generated predicates match actual documents.

In [10]:
EXAMPLES_PER_TYPE = 0
schema_text = render_schema_catalog(schemas, examples_per_type=EXAMPLES_PER_TYPE)
print(schema_text)

Schema observations from inspected documents; not full-collection guarantees.
Notation: . denotes nested fields; [] denotes array elements (not MongoDB query syntax).
Bracket-quoted names denote literal field names, e.g. ["address.city"].
Missing counts refer to immediate parent objects; null is a present value.

Collection: "businesses"
Documents inspected: 2
Fields:
- address: object; missing in 0/2 parent objects
- address.city: string; missing in 1/2 parent objects
- business_id: string; missing in 0/2 parent objects
- categories: array; missing in 0/2 parent objects
- categories[]: string
- name: string; missing in 0/2 parent objects
- stars: float | null; missing in 0/2 parent objects

Collection: "reviews"
Documents inspected: 2
Fields:
- business_id: string; missing in 0/2 parent objects
- rating: integer; missing in 0/2 parent objects
- text: string; missing in 0/2 parent objects
- user_id: string; missing in 0/2 parent objects

Collection: "users"
Documents inspected: 2
Field

## 4. Choose the research task and preview the prompt

`p3` requests find queries; `p4` requests simple aggregation pipelines. The other templates explore more complex queries. Each template also declares the query kind expected by the parser.

`NUM_QUERIES` is the total requested; `BATCH_SIZE` is queries per model call. Here we request batches of two and one. All queries should start at `businesses`, while the other schemas remain available for joins. Relationship hints are supplied explicitly: schema inspection alone does not infer foreign keys.

In [ ]:
PROMPT_ID = "p4"
NUM_QUERIES = 3
BATCH_SIZE = 2
TARGET_COLLECTION = "businesses"
RELATIONSHIP_HINTS = "reviews.business_id references businesses.business_id. reviews.user_id references users.user_id."
template = PROMPTS[PROMPT_ID]
preview = template.render(
    schema_text,
    num_queries=min(NUM_QUERIES, BATCH_SIZE),
    target_collection=TARGET_COLLECTION,
    relationship_hints=RELATIONSHIP_HINTS,
)
print(f"Prompt: {PROMPT_ID}; expected query kind: {template.query_kind}\n")
print(preview)

Prompt: p3; expected query kind: find

Task
Generate a MongoDB find query suitable for performance benchmarking.

Schema catalogue
Schema observations from inspected documents; not full-collection guarantees.
Notation: . denotes nested fields; [] denotes array elements (not MongoDB query syntax).
Bracket-quoted names denote literal field names, e.g. ["address.city"].
Missing counts refer to immediate parent objects; null is a present value.

Collection: "businesses"
Documents inspected: 2
Fields:
- address: object; missing in 0/2 parent objects
- address.city: string; missing in 1/2 parent objects
- business_id: string; missing in 0/2 parent objects
- categories: array; missing in 0/2 parent objects
- categories[]: string
- name: string; missing in 0/2 parent objects
- stars: float | null; missing in 0/2 parent objects

Collection: "reviews"
Documents inspected: 2
Fields:
- business_id: string; missing in 0/2 parent objects
- rating: integer; missing in 0/2 parent objects
- text: strin

## 5. Configure model access

Ollama uses the same MongoDB system instruction as hosted API and subscription routes. `think=False` disables thinking for the tested local Qwen model; omit it to use the model default. Parameters are model-specific—do not carry this setting blindly to other providers.

To use your authenticated subscription instead, replace the constructor with:

```python
model = LiteLLMModel("chatgpt/gpt-6.1-sol", parameters={"reasoning_effort": "low"})
```

Complete subscription device login in a terminal first, as described in `docs/models.md`. For a paid API route, choose a supported provider/model and configure its credentials in `.env`. Constructing a model does not send a request.

In [5]:
model = LiteLLMModel(
    "ollama_chat/qwen3.6:latest",
    api_base="http://localhost:11434",
    timeout_seconds=120,
    parameters={"think": False},
)
print("Model:", model.model)
print("System instruction:", model.system_instructions)

Model: ollama_chat/qwen3.6:latest
System instruction: You generate MongoDB queries. Follow the user's query instructions and response format exactly.


## 6. Generate batches — this cell calls the model

`generate_queries` snapshots the schemas and validates configuration immediately. Calls happen as we iterate. Each batch gets an independent prompt; earlier answers are not fed back to the model.

Keep the records as they arrive so completed work remains inspectable if a later call fails. A model-call error is recorded and stops generation. Invalid JSON, short batches, and individual query errors are observations; they do not trigger repair or replacement. Rerunning this cell makes new requests and replaces `batches`.

In [ ]:
batches = []
for batch in generate_queries(
    model,
    template,
    schemas,
    database="research",
    num_queries=NUM_QUERIES,
    batch_size=BATCH_SIZE,
    target_collection=TARGET_COLLECTION,
    relationship_hints=RELATIONSHIP_HINTS,
    examples_per_type=EXAMPLES_PER_TYPE,
):
    batches.append(batch)
    if batch.error is not None:
        print(f"Batch {batch.batch_index}: model call failed ({type(batch.error).__name__})")
    else:
        print(
            f"Batch {batch.batch_index}: requested {batch.requested_count}, "
            f"received {batch.parsed.actual_count}, "
            f"target violations {batch.target_violations}"
        )
print(f"Retained {len(batches)} attempted batches.")

Batch 0: requested 2, received 2, target violations ()
Batch 1: requested 1, received 1, target violations ()
Retained 2 attempted batches.


## 7. Inspect raw responses and parsing outcomes

The raw model text and parsed BONDI objects serve different purposes. Raw text explains failures; valid objects are ready for later workload construction. `invalid_json` means decoding failed, while `invalid_query_shape` means decoded content violated the response contract.

A structurally valid query can still have invalid MongoDB operators, be unsafe to execute, or return no documents. Target violations are tracked separately and must be excluded from execution.

In [ ]:
candidate_rows = []
for batch in batches:
    if batch.error is not None:
        continue  # the failed call has no model response to parse
    print(f"\nBatch {batch.batch_index} raw response:\n{batch.response.text}")
    if batch.parsed.error is not None:
        print("Batch parse failure:", batch.parsed.error)
    for candidate in batch.parsed.candidates:
        status = (
            candidate.error.status
            if candidate.error
            else "target_violation"
            if candidate.position in batch.target_violations
            else "parsed"
        )
        candidate_rows.append(
            {
                "batch_index": batch.batch_index,
                "position": candidate.position,
                "status": status,
                "query_type": type(candidate.query).__name__ if candidate.query else None,
                "error": candidate.error.message if candidate.error else None,
            }
        )
display(pd.DataFrame(candidate_rows))


Batch 0 raw response:
{
  "queries": [
    {
      "collection": "businesses",
      "filter": {
        "stars": {
          "$gt": 3.5
        }
      },
      "projection": {
        "name": 1,
        "address.city": 1,
        "stars": 1
      },
      "sort": [
        [
          "stars",
          -1
        ]
      ],
      "limit": 10
    },
    {
      "collection": "businesses",
      "filter": {
        "categories": {
          "$elemMatch": {
            "$regex": "Restaurant"
          }
        }
      },
      "projection": {
        "name": 1,
        "categories": 1
      },
      "skip": 0,
      "limit": null
    }
  ]
}

Batch 1 raw response:
{
  "queries": [
    {
      "collection": "businesses",
      "filter": {},
      "limit": null
    }
  ]
}


,batch_index,position,status,query_type,error
0,0,0,parsed,MongoFindQuery,None
1,0,1,parsed,MongoFindQuery,None
2,1,0,parsed,MongoFindQuery,None


## 8. Inspect usage and build an offline workload

Usage belongs to a model call, not to each query. Reasoning/cache counts may already be included in totals; missing breakdowns stay unknown. Subscription calls have no per-token API estimate, while Ollama's zero estimate excludes local compute costs.

The workload below includes only structurally parsed candidates with the requested target. Preserve `(batch_index, position)` alongside it, because filtering changes workload positions. This is an **offline inspection workload**, not execution approval: the later evaluator must check read-only policy and MongoDB outcomes.

In [ ]:
usage_rows = []
entries = []
candidate_mapping = []
for batch in batches:
    if batch.response is None:
        continue
    r = batch.response
    usage_rows.append(
        {
            "batch_index": batch.batch_index,
            "input_tokens": r.input_tokens,
            "output_tokens": r.output_tokens,
            "reasoning_tokens": r.reasoning_tokens,
            "elapsed_ms": r.elapsed_ms,
            "finish_reason": r.finish_reason,
            "billing_mode": r.billing_mode,
            "estimated_cost_usd": r.estimated_cost_usd,
        }
    )
    for candidate in batch.parsed.candidates:
        if candidate.query is not None and candidate.position not in batch.target_violations:
            entries.append(WorkloadEntry(candidate.query))
            candidate_mapping.append((batch.batch_index, candidate.position))
display(pd.DataFrame(usage_rows))
# BONDI workloads require at least one entry; keep the all-failed case explicit.
workload = Workload(entries) if entries else None
print("Accepted structural candidates:", len(entries))
print("Workload position → (batch index, candidate position):", candidate_mapping)
if entries:
    print("First query as BONDI JSON:")
    print(entries[0].query.to_json())

,batch_index,input_tokens,output_tokens,reasoning_tokens,elapsed_ms,finish_reason,billing_mode,estimated_cost_usd
0,0,640,209,None,9556.240375,stop,provider,0.0
1,1,640,40,None,1008.474792,stop,provider,0.0


Accepted structural candidates: 3
Workload position → (batch index, candidate position): [(0, 0), (0, 1), (1, 0)]
First query as BONDI JSON:
{"format_version":1,"query_kind":"mongodb.find","target":{"database":"research","collection":"businesses"},"filter":{"stars":{"$gt":{"$numberDouble":"3.5"}}},"projection":{"name":1,"address.city":1,"stars":1},"sort":[["stars",-1]],"skip":0,"limit":10}


## What to try next

- Switch `p3` to `p4` and inspect aggregation pipelines.
- Compare `BATCH_SIZE = 1` with larger batches, retaining duplicates as observations.
- Include two example values per type and inspect the changed schema text.
- Remove the target restriction with `TARGET_COLLECTION = None`.

After changing configuration, rerun the affected cells through generation and inspection. Keep model, prompt, schema, and reasoning settings fixed when studying one variable. Results here are in memory; Yanex artifact persistence and MongoDB evaluation come later.